# Train PPO

Train a categorical policy on `CartPole-v1` using the public AprendeRL API. PPO uses a clipped surrogate and repeated minibatch Adam steps.

$$L^{CLIP}=\mathbb E_t[\min(\rho_t\hat A_t,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)\hat A_t)].$$

Here $\rho_t=\pi_\theta(a_t\mid s_t)/\pi_{old}(a_t\mid s_t)$ compares current and collecting-policy action probabilities, $\hat A_t$ is the GAE advantage, and $\mathbb E_t$ averages over rollout samples. The clipping range is $\epsilon$; clipping does not impose a hard KL bound.

See the [algorithm guide](../docs/algorithms/ppo.md). This CPU demonstration uses separate actor and critic networks.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import PPO, PPOConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "CartPole-v1"
TOTAL_TIMESTEPS = 20_000
EVAL_EPISODES = 5
SEED = 7

## Configure and train

GAE bootstraps through time-limit truncation but stops its trace at every episode boundary. Complete rollouts trigger updates; a partial rollout is retained for a subsequent `learn` call. Sampling the policy provides exploration. Episode returns are recorded by the shared training loop.

In [ ]:
config = PPOConfig(
    n_steps=256,
    gamma=0.99,
    gae_lambda=0.95,
    value_learning_rate=1e-3,
    learning_rate=3e-4,
    batch_size=64,
    n_epochs=10,
    clip_range=0.2,
    seed=SEED,
)
env = gym.make(ENV_ID)
try:
    agent = PPO(env, config=config, device="cpu")
    agent.learn(total_timesteps=TOTAL_TIMESTEPS)
finally:
    env.close()

## Inspect episode returns

Plot undiscounted episode returns and a moving average over up to 20 episodes. A short run illustrates the API; compare multiple seeds before drawing conclusions about learning performance.

In [ ]:
returns = np.asarray(agent.episode_returns)
plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
if len(returns):
    window = min(20, len(returns))
    moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")
    plt.plot(
        np.arange(window - 1, len(returns)),
        moving_average,
        label=f"{window}-episode average",
    )
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"PPO training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Evaluate the deterministic policy

Use a separate rendered environment and select the most probable action. Display the final frame inline; switch to `render_mode="human"` for a live desktop window and remove the `imshow` cell.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="rgb_array")
try:
    result = evaluate_policy(
        agent,
        evaluation_env,
        episodes=EVAL_EPISODES,
        deterministic=True,
        seed=SEED + 1000,
    )
    final_frame = evaluation_env.render()
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(final_frame)
plt.axis("off")
plt.show()